# 06 — Human feedback loop

**Otázka:** LLM-judge říká `no_hallucinated_products = 1.0`. Důvěřujeme mu na 100 %?

Ne. Online evaluators měří, co jsi *uměl popsat v promptu*. Lidi měří, co je *doopravdy* dobře. Potřebuješ obojí.

Tenhle notebook stavi celý loop:

```
  end-user 👎 ┐
  evaluator ❌ ├──► triage ──► annotation queue ──► human review ──► dataset growth
  random 5%  ┘                                                       agreement report
```

Předpoklady:
- Máš v projektu `kosik-workshop` nějaký traffic za posledních 24h. Pokud ne, pust `uv run python scripts/run_demo.py --scenario baseline`.
- Volitelně `uv run python scripts/chat_app.py --user user-petr` a klikni 5× 👎/👍 — to je end-user signál.
- Dataset `kosik-eval-golden` existuje (jinak `python scripts/seed_eval_dataset.py`).

In [ ]:
from kosik_workshop.config import load_env
load_env()

## 1. Triage policy

Tři pravidla, která skript `seed_annotation_queue.py` plní do queue `kosik-human-review`:

| Pravidlo | Co flaguje | Proč |
|---|---|---|
| `thumbs_down` | `feedback.user_thumbs = 0` | Uživatel sám řekl, že je to špatně |
| `eval_flagged` | `online.heuristic_quality = 0` nebo `hallucination = 1` | Online evaluator se chytil podezření |
| `random` | Uniformní vzorek N % | Prevence survivorship bias — chytá silent failures |

Pravidla se kombinují (set union). Co už v queue je, se znovu nepushuje.

In [ ]:
# Dry run nejdřív — uvidíš plán bez writes.
!uv run python scripts/seed_annotation_queue.py --hours 24 --random-pct 5 --dry-run

In [ ]:
# Pust naostro — vytvoří queue (pokud neexistuje) a pushne kandidáty.
!uv run python scripts/seed_annotation_queue.py --hours 24 --random-pct 5

## 2. Human review v LangSmith UI

Tohle je **jediný manuální krok** v celém loopu. Otevři:

1. [https://eu.smith.langchain.com](https://eu.smith.langchain.com) → **Annotation queues** v levém menu
2. `kosik-human-review` → **Start annotating**
3. Pro každý trace vyplň 3 booleany + krátký komentář (root cause):
   - **`correct_tools`** — agent zavolal správnou kombinaci tools?
   - **`helpful`** — odpověd je věcná, řeší dotaz uživatele?
   - **`safe`** — žádná halucinace, alergeny správně?
4. Submit → další trace.

**Tip pro workshop:** rozdej 2 lidem stejných 5 traces a porovnej anotace. IAA (inter-annotator agreement) málokdy bývá 100 % — proto rubrika.

**Anotuj alespoň 5–8 traces** než pojedeš dál. Kdo má queue prázdnou, ať pustí buňku níž (simulace 👎).

In [ ]:
# Volitelné: simuluj end-user 👎 na pár existujících traces, ať je v queue víc kandidátů.
# Reálně tohle posílá tvoje aplikace — tady to faké pro účely workshopu.
import random
from datetime import datetime, timedelta, timezone
from langsmith import Client

client = Client()
since = datetime.now(timezone.utc) - timedelta(hours=24)
candidates = list(client.list_runs(
    project_name='kosik-workshop',
    start_time=since,
    is_root=True,
    limit=50,
))
rng = random.Random(1)
for run in rng.sample(candidates, k=min(5, len(candidates))):
    client.create_feedback(
        run_id=run.id,
        key='user_thumbs',
        score=0,
        comment='[simulated 👎 for workshop]',
    )
    print(f'  👎 {str(run.id)[:8]}  {run.start_time:%H:%M}')
print('\nTeď spust seed_annotation_queue.py znovu — uvidíš thumbs_down rule v akci.')

## 3. Eval-vs-human agreement

Po anotacích spočítáme, jak moc tvůj LLM-judge *fituje* na lidech. Když je agreement < 70 %, judge měří jiný koncept než lidi → akční položka pro úpravu judge promptu (případně dataset rozšíření).

In [ ]:
!uv run python scripts/eval_human_agreement.py --queue kosik-human-review

## 4. Promote anotace do datasetu

Anotace nejsou pro pocit. Promote skript vezme:

- **Good** examples (`correct_tools=1 AND helpful=1 AND safe=1`) → přidá jako pozitivní cases.
- **Bad** examples (`helpful=0 OR safe=0`) → přidá jako **regression cases** s `is_regression=True` a human comment.

Bad examples jsou často cennější — chrání před opětovným výskytem stejného problému.

In [ ]:
!uv run python scripts/promote_annotations_to_dataset.py --dry-run

In [ ]:
!uv run python scripts/promote_annotations_to_dataset.py

In [ ]:
# Sanity check: dataset narostl?
from langsmith import Client
from kosik_workshop.evals.dataset import DATASET_NAME

client = Client()
ds = next(iter(client.list_datasets(dataset_name=DATASET_NAME)))
examples = list(client.list_examples(dataset_id=ds.id))
by_category = {}
for ex in examples:
    cat = (ex.outputs or {}).get('category', 'unknown')
    by_category[cat] = by_category.get(cat, 0) + 1
print(f'Total examples: {len(examples)}\n')
for cat, n in sorted(by_category.items(), key=lambda x: -x[1]):
    print(f'  {cat:<40} {n:>3}')

## 5. Re-run eval gate na rozšířeném datasetu

Dataset roste živě. Některé varianty, které dřív vypadaly OK, můžou propadnout na nově přidaných regression cases. **To je celý smysl loopu.**

Pokud chceš plnohodnotně otestovat, pust `06_ab_prompts.ipynb` znovu — uvidíš, jestli `v3_loud_allergens` zvládá nové edge-cases z lidských anotací.

In [ ]:
# Quick check: spust default A/B set na rozšířeném datasetu.
# Trvá ~3–5 min × 3 varianty.
# !uv run python scripts/run_ab_eval.py --variants v1_baseline,v2_strict_citations --group post-human-review

## 6. Co běží autonomně, co potřebuje člověka

```
Auto:                              Human:
─ Online eval na 100 % traffic     ─ Anotace v queue (jediný manuál krok)
─ Sampling do queue (3 pravidla)   ─ Triage rozhodnutí: zvedat threshold?
─ End-user thumbs route do queue   ─ Schvalování dataset growth (PR review)
─ Promote dataset (skript)         ─ Prompt iterace na patterns z comments
─ Eval gate na PR (CI)
─ Drift alert na human-pass-rate
```

**Závěr:** Human-in-the-loop neznamená člověk dělá všechno. Znamená lidi jsou *ground-truth oracle*, který kalibruje ostatní automatizovanou infrastrukturu. Cíl není odstranit lidi — cíl je, aby drahý lidský čas šel **jen** tam, kde má smysl.